# 04 - Evaluation

Compare base model vs fine-tuned model on all financial tasks.

**Prerequisites:** Run `03_training_qlora.ipynb` first.

In [ ]:
%%capture
!pip install unsloth rouge_score bert_score

In [ ]:
import os, json
os.chdir('/kaggle/working/fingpt-qlora')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Test Data & Prepare
Load data first, then load models one at a time to avoid OOM on T4.

In [ ]:
from unsloth import FastLanguageModel

EXPERIMENT_NAME = "v1_baseline"  # Change to match your training run

with open('data/splits/test.json') as f:
    test_data = json.load(f)

print(f"Test set: {len(test_data)} examples")
task_counts = Counter(ex.get('task_type', 'unknown') for ex in test_data)
for task, count in sorted(task_counts.items()):
    print(f"  {task}: {count}")


def generate_response(model, tokenizer, messages, max_new_tokens=512):
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)


def build_messages(conversations):
    messages = []
    for turn in conversations:
        role = turn['from']
        if role == 'gpt':
            role = 'assistant'
        messages.append({'role': role, 'content': turn['value']})
    return messages[:-1]

## 2. Generate Base Model Predictions (then free GPU memory)

In [ ]:
# Load base model
print("Loading base model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print("Base model loaded.")

# Generate base predictions
base_predictions = []
for i, example in enumerate(test_data):
    messages = build_messages(example['conversations'])
    pred = generate_response(base_model, base_tokenizer, messages)
    base_predictions.append(pred)
    if (i + 1) % 20 == 0:
        print(f"Base model: {i+1}/{len(test_data)}")

# Free base model GPU memory
del base_model
del base_tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"\nBase predictions done: {len(base_predictions)}. GPU memory freed.")

## 3. Generate Fine-tuned Model Predictions

In [ ]:
# Load fine-tuned model
ft_model_path = f"outputs/{EXPERIMENT_NAME}/lora_adapter"
print(f"Loading fine-tuned model from {ft_model_path}...")
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ft_model_path,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)
print("Fine-tuned model loaded.")

# Generate fine-tuned predictions
ft_predictions = []
for i, example in enumerate(test_data):
    messages = build_messages(example['conversations'])
    pred = generate_response(ft_model, ft_tokenizer, messages)
    ft_predictions.append(pred)
    if (i + 1) % 20 == 0:
        print(f"Fine-tuned model: {i+1}/{len(test_data)}")

# Free fine-tuned model GPU memory
del ft_model
del ft_tokenizer
torch.cuda.empty_cache()
gc.collect()
print(f"\nFine-tuned predictions done: {len(ft_predictions)}.")

# Assemble results
results = []
for i, example in enumerate(test_data):
    results.append({
        'reference': example['conversations'][-1]['value'],
        'base_prediction': base_predictions[i],
        'ft_prediction': ft_predictions[i],
        'task_type': example.get('task_type', 'unknown'),
    })

os.makedirs('results/eval_outputs', exist_ok=True)
with open(f'results/eval_outputs/{EXPERIMENT_NAME}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved {len(results)} results.")

## 4. Compute Metrics

In [ ]:
import re

def extract_sentiment(text):
    match = re.search(r'\*\*Sentiment:\s*(\w+)\*\*', text, re.IGNORECASE)
    if match:
        label = match.group(1).capitalize()
        if label in ('Positive', 'Negative', 'Neutral'):
            return label
    text_lower = text.lower()
    if 'positive' in text_lower and 'negative' not in text_lower:
        return 'Positive'
    if 'negative' in text_lower and 'positive' not in text_lower:
        return 'Negative'
    return 'Neutral'


def compute_sentiment_metrics(predictions, references):
    pred_labels = [extract_sentiment(p) for p in predictions]
    ref_labels = [extract_sentiment(r) for r in references]
    correct = sum(1 for p, r in zip(pred_labels, ref_labels) if p == r)
    return {'accuracy': correct / len(ref_labels), 'correct': correct, 'total': len(ref_labels)}


def compute_rouge_l(predictions, references):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = [scorer.score(r, p)['rougeL'].fmeasure for p, r in zip(predictions, references)]
    return {'rouge_l_f1': np.mean(scores), 'std': np.std(scores)}

In [ ]:
# Compute metrics by task type
metrics_table = []

for task_type in sorted(set(r['task_type'] for r in results)):
    task_results = [r for r in results if r['task_type'] == task_type]
    if not task_results:
        continue
    
    refs = [r['reference'] for r in task_results]
    base_preds = [r['base_prediction'] for r in task_results]
    ft_preds = [r['ft_prediction'] for r in task_results]
    
    if task_type == 'sentiment':
        base_m = compute_sentiment_metrics(base_preds, refs)
        ft_m = compute_sentiment_metrics(ft_preds, refs)
        metrics_table.append({
            'Task': task_type,
            'Metric': 'Accuracy',
            'Base': f"{base_m['accuracy']:.3f}",
            'Fine-tuned': f"{ft_m['accuracy']:.3f}",
            'Delta': f"{ft_m['accuracy'] - base_m['accuracy']:+.3f}",
        })
    
    # ROUGE-L for all tasks
    try:
        base_rouge = compute_rouge_l(base_preds, refs)
        ft_rouge = compute_rouge_l(ft_preds, refs)
        metrics_table.append({
            'Task': task_type,
            'Metric': 'ROUGE-L',
            'Base': f"{base_rouge['rouge_l_f1']:.3f}",
            'Fine-tuned': f"{ft_rouge['rouge_l_f1']:.3f}",
            'Delta': f"{ft_rouge['rouge_l_f1'] - base_rouge['rouge_l_f1']:+.3f}",
        })
    except Exception as e:
        print(f"ROUGE-L failed for {task_type}: {e}")

df_metrics = pd.DataFrame(metrics_table)
print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(df_metrics.to_string(index=False))

# Save
df_metrics.to_markdown('results/comparison_table.md', index=False)
print(f"\nSaved to results/comparison_table.md")

## 5. Visualize Results

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))

sentiment_metrics = [m for m in metrics_table if m['Metric'] == 'Accuracy']
if sentiment_metrics:
    labels = [m['Task'] for m in sentiment_metrics]
    base_vals = [float(m['Base']) for m in sentiment_metrics]
    ft_vals = [float(m['Fine-tuned']) for m in sentiment_metrics]
    
    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width/2, base_vals, width, label='Base Model', color='#e74c3c', alpha=0.8)
    ax.bar(x + width/2, ft_vals, width, label='Fine-tuned', color='#2ecc71', alpha=0.8)
    ax.set_ylabel('Score')
    ax.set_title(f'Model Comparison - {EXPERIMENT_NAME}')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 1)
    
    for i, (b, f) in enumerate(zip(base_vals, ft_vals)):
        ax.text(i - width/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)
        ax.text(i + width/2, f + 0.01, f'{f:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'results/figures/comparison_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()

# Bar chart comparison
os.makedirs('results/figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))

sentiment_metrics = [m for m in metrics_table if m['Metric'] == 'Accuracy']
if sentiment_metrics:
    labels = [m['Task'] for m in sentiment_metrics]
    base_vals = [float(m['Base']) for m in sentiment_metrics]
    ft_vals = [float(m['Fine-tuned']) for m in sentiment_metrics]
    
    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width/2, base_vals, width, label='Base Model', color='#e74c3c', alpha=0.8)
    ax.bar(x + width/2, ft_vals, width, label='Fine-tuned', color='#2ecc71', alpha=0.8)
    ax.set_ylabel('Score')
    ax.set_title(f'Model Comparison - {EXPERIMENT_NAME}')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 1)
    
    for i, (b, f) in enumerate(zip(base_vals, ft_vals)):
        ax.text(i - width/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)
        ax.text(i + width/2, f + 0.01, f'{f:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'results/figures/comparison_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()

In [ ]:
from IPython.display import display, HTML

# Show 5 examples with biggest improvement
for task_type in sorted(set(r['task_type'] for r in results)):
    task_results = [r for r in results if r['task_type'] == task_type][:3]
    
    print(f"\n{'='*70}")
    print(f"Task: {task_type}")
    print(f"{'='*70}")
    
    for i, r in enumerate(task_results):
        print(f"\n--- Example {i+1} ---")
        print(f"Reference: {r['reference'][:200]}...")
        print(f"\nBase Model:\n{r['base_prediction'][:300]}...")
        print(f"\nFine-tuned:\n{r['ft_prediction'][:300]}...")

## 7. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap CI for sentiment accuracy
sentiment_results = [r for r in results if r['task_type'] == 'sentiment']
if sentiment_results:
    refs = [r['reference'] for r in sentiment_results]
    ft_preds = [r['ft_prediction'] for r in sentiment_results]
    
    rng = np.random.RandomState(42)
    n = len(refs)
    boot_scores = []
    
    for _ in range(1000):
        idx = rng.choice(n, size=n, replace=True)
        boot_refs = [refs[i] for i in idx]
        boot_preds = [ft_preds[i] for i in idx]
        m = compute_sentiment_metrics(boot_preds, boot_refs)
        boot_scores.append(m['accuracy'])
    
    ci_lower = np.percentile(boot_scores, 2.5)
    ci_upper = np.percentile(boot_scores, 97.5)
    
    print(f"Sentiment Accuracy (Fine-tuned):")
    print(f"  Mean: {np.mean(boot_scores):.3f}")
    print(f"  95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
    print(f"  Std: {np.std(boot_scores):.3f}")
else:
    print("No sentiment examples in test set.")